In [ ]:
# !pip install psycopg2-binary -q

### This script will update the last two years of 'logistics_ND_agg_stock_disp' table  

-Update table partitions

## Aggregations for logistics_ND table (proportion)

This script processes and aggregates the 'logistics_ND' dataset to prepare it for dashboard visualization in Superset.  
The full aggregation has been computed and stored in the table, now we only process the last year.  

It is possible to update specific or all years by setting the variable `target_years`, see the cell with tag: _# Target year selection_

In [ ]:
import pandas as pd 
import polars as pl 
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace, current_run
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc
from pnlp_utils import log_msg, push_dataframe_to_db, check_partition_exist, upsert_partition, create_partition_indexes

In [ ]:
# Parameters
YEAR = 2026

In [ ]:
columns_expected = [
    'aggregation_spatiale',
    'aggregation_temporelle',
    'period',
    'annee',
    'geo',
    'indicateur_name',
    'de_name',
    'province',
    'zone_de_sante',
    'aire_de_sante',
    'absolute_indicateur_value',
    'taux_indicateur_value',
    'indicateur_num',
    'indicateur_den'
]


In [ ]:
# Aggregations logic for indicators_nd table 
def run_logistics_nd_aggregations(df_year: pd.DataFrame) -> pl.DataFrame:

    # Spatial & temporal hierarchies
    spatial_hierarchy = {
        "Pays": ["LEVEL_1_NAME"],
        "Province": ["LEVEL_1_NAME", "LEVEL_2_NAME"],
        "Zone": ["LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME"],
        "Aire": ["LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME", "LEVEL_4_NAME"],
    }
    
    temporal_levels = {
        # "Année": "Year_period",
        # "Semestre": "Semester_period",
        # "Trimestre": "Quarter_period",
        "Mois": "Month_period",
    }
    
    standard_cols = [
        "Aggregation_spatiale", 
        "Aggregation_temporelle",
        "LEVEL_1_NAME", 
        "LEVEL_2_NAME", 
        "LEVEL_3_NAME", 
        "LEVEL_4_NAME",
        "Indicateur_name", 
        "Period",
        "Annee", 
        "Indicateur_num", 
        "Indicateur_den",
        "Absolute_indicateur_value", 
        "Taux_indicateur_value", 
        "Geo", 
        "DE_NAME"
    ]

 
    # Add year and period columns    
    print("Creating period columns..")
    
    # Convert pandas DataFrame to Polars (faster)
    df_pl = pl.from_pandas(df_year)   
    df_pl = df_pl.with_columns(pl.col("Agg_period").str.strptime(pl.Date, "%b %Y").alias("Agg_date"))
    
    df_pl = df_pl.with_columns(    
        pl.col("Annee").cast(pl.Utf8).alias("Year_period"),
        (pl.col("Agg_date").dt.year().cast(pl.Utf8) + " S" + (((pl.col("Agg_date").dt.month() - 1) // 6 + 1).cast(pl.Utf8))).alias("Semester_period"),
        (pl.col("Agg_date").dt.year().cast(pl.Utf8) + " T" + (((pl.col("Agg_date").dt.month() - 1) // 3 + 1).cast(pl.Utf8))).alias("Quarter_period"),
        pl.col("Agg_date").dt.strftime("%Y-%m").alias("Month_period"),
        pl.col("DE_NAME").str.replace(r"\s(?:Number|Zero or Positive Int)$", "", literal=False)
    )

    agg_list_pl = []
    
    for spatial_name, spatial_cols in spatial_hierarchy.items():
        print(f"Processing → Spatial: {spatial_name}")
        for temporal_name, temporal_col in temporal_levels.items():
                
            group_cols = spatial_cols + ["Indicateur_name", "DE_NAME", temporal_col, "Annee"]
            
            # Polars groupby + aggregation 
            agg_df = (
                df_pl.group_by(group_cols, maintain_order=True)
                     .agg([
                         pl.sum("Indicateur_num").alias("Indicateur_num"),
                         pl.sum("Indicateur_den").alias("Indicateur_den")
                     ])
            )
    
            # Compute absolute value and rate
            agg_df = agg_df.with_columns([
                pl.col("Indicateur_num").alias("Absolute_indicateur_value"),
                (pl.when(pl.col("Indicateur_den") != 0)
                   .then(pl.col("Indicateur_num") / pl.col("Indicateur_den"))
                   .otherwise(None)
                ).alias("Taux_indicateur_value")
            ])

            # Ensure all spatial columns exist with correct type (Utf8)
            for col in ["LEVEL_1_NAME", "LEVEL_2_NAME", "LEVEL_3_NAME", "LEVEL_4_NAME"]:
                if col not in agg_df.columns:
                    agg_df = agg_df.with_columns(pl.lit(None, dtype=pl.Utf8).alias(col))
 
            # Metadata 
            agg_df = agg_df.with_columns([
                pl.lit(spatial_name).alias("Aggregation_spatiale"),
                pl.lit(temporal_name).alias("Aggregation_temporelle"),
                pl.col(temporal_col).alias("Period"),
                pl.col("Indicateur_name")
                  .str.replace_all("_", " ")   # replace all underscores
                  .str.to_lowercase()  
                  .str.to_titlecase()                 
                  .alias("Indicateur_name")
            ])
                    
            # Drop the temporal column
            agg_df = agg_df.drop(temporal_col)

            # Set "Geo" column
            agg_df = agg_df.with_columns(
                pl.coalesce(["LEVEL_4_NAME", "LEVEL_3_NAME", "LEVEL_2_NAME", "LEVEL_1_NAME"]).alias("Geo")
            )
 
            # Reorder agg_df and append
            agg_df = agg_df.select([col for col in standard_cols if col in agg_df.columns])
            agg_list_pl.append(agg_df)
        
    # Concatenate all results 
    df_year_agg_pl = pl.concat(agg_list_pl)       
    df_year_agg_pl = df_year_agg_pl.rename({
        "LEVEL_2_NAME": "Province",
        "LEVEL_3_NAME": "Zone_de_sante",
        "LEVEL_4_NAME": "Aire_de_sante",
    })
    
    df_year_agg_pl = df_year_agg_pl.drop("LEVEL_1_NAME")  # drop
    
    df_year_agg_pl = df_year_agg_pl.with_columns([
        pl.col("Province").fill_null("N/D"), 
        pl.col("Zone_de_sante").fill_null("N/D"), 
        pl.col("Aire_de_sante").fill_null("N/D")
    ])

    df_year_agg_pl = df_year_agg_pl.rename({c: c.lower() for c in df_year_agg_pl.columns})  # cols to lower
    
    return df_year_agg_pl


### Start processing

In [ ]:
# table names
source_table = "logistics_ND"
target_table = "logistics_ND_table_produits_traceurs" 
indicators_selection = ["CMM", "Stock_Disp", "Mois_stock"]  # retrieve data only for these indicators

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# Load available years
years_available = pd.read_sql(text(f'SELECT distinct "Annee" FROM public."{source_table}"'), engine).Annee.tolist()
years_available = [int(y) for y in sorted(years_available)] 
print(f"Years available in source: {years_available}")

In [ ]:
# Target year selection
target_years = [YEAR]
# target_years = years_available[-1:]  # Select current year
log_msg(f"Aggregating '{target_table}' for years: {target_years}")

In [ ]:
# loop over selected years
for target_year in target_years:
    
    log_msg(f"Running for year: {target_year}")
 
    log_msg(f"Loading data from '{source_table}' for year {target_year}")    
    query = f"""SELECT * FROM public."{source_table}" WHERE "Annee" = %(year)s AND "Indicateur_name" = ANY(%(indicators)s)"""
    params = {"year": f"{target_year}", "indicators": indicators_selection}
    
    df_year = pd.read_sql(query, con=engine, params=params)
    print(f".. ready, shape: {df_year.shape}")
    
    df_year_agg = run_logistics_nd_aggregations(df_year)
    assert df_year_agg.height == df_year_agg.unique().height, "Duplicated values found!" # quick check!

    # Rita prefer these names ;) 
    df_year_agg = df_year_agg.with_columns(
        pl.col("indicateur_name").replace(
            {
                "Mois Stock": "Mois de Stock Disponible",
                "Stock Disp": "Stock Disponible Utilisable",
                "Cmm": "Consommation Mensuelle Moyenne"
            }
        )
    )
    
    print("Check expected columns..", end=" ")
    df_year_agg = df_year_agg.select(columns_expected)
    print("OK!")

    log_msg("Pushing aggregated data to db..")
    tmp_table = f"{target_table}_tmp"
    push_dataframe_to_db(
        df=df_year_agg, 
        table_name=tmp_table, 
        engine=engine, 
        push_behaviour="replace",
        chunksize=500000
    ) 

    # update partition
    upsert_partition(engine, target_table, tmp_table, target_year)
    
    log_msg("Creating indexes..")
    index_columns = ["aggregation_spatiale", "aggregation_temporelle", "indicateur_name", "de_name", "geo"]
    create_partition_indexes(engine, target_table, target_year, index_columns)

    # clean
    del df_year_agg, df_year
    collected = gc.collect()
    print(collected)
 

In [ ]:
log_msg("Finished!")